### Middleware

**Middleware** is an intermediate layer in a LangChain agent that allows us to intercept and control the agent's execution before, during, or after model and tool calls. It can be used to modify inputs and outputs, manage conversation history, summarize long conversations, handle errors, monitor execution, enforce limits, and add custom logic without changing the core agent or model. Middleware is especially useful for building production-ready agents because it provides better **control, reliability, security, and observability**.

| Type / Middleware            | Use                                                                   |
| ---------------------------- | --------------------------------------------------------------------- |
| **Summarization Middleware** | Summarizes long conversation history when token limits are reached    |
| **Human-in-the-Loop**        | Pauses execution and asks for human approval before sensitive actions |
| **Model Call Limit**         | Limits the number of times an agent can call the LLM                  |
| **Tool Call Limit**          | Limits the number of tool executions                                  |
| **Model Fallback**           | Uses another model when the primary model fails                       |
| **PII / Data Redaction**     | Detects and removes or masks sensitive information                    |
| **Prompt Modification**      | Modifies or enriches prompts before they reach the model              |
| **Message Filtering**        | Filters or removes unwanted messages from the conversation            |
| **Guardrails**               | Enforces rules on model inputs and outputs                            |
| **Custom Middleware**        | Allows developers to implement application-specific logic             |



In [3]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver

# Loads environment variables from the .env file.
# This allows the GROQ_API_KEY stored in .env to be
# automatically available to ChatGroq.
load_dotenv()


# Creates the Groq chat model that will be used by the agent.
# The model is responsible for understanding the user query
# and generating the final response.
model = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)


# Creates summarization middleware.
#
# Middleware runs in the agent's execution flow and can
# automatically manage the conversation history.
#
# trigger=("tokens", 4000):
# When the conversation reaches approximately 4000 tokens,
# the middleware triggers summarization so that the complete
# conversation does not continue growing indefinitely.
#
# keep=("messages", 10):
# After summarization, the middleware keeps the most recent
# 10 messages while older messages can be represented by
# the generated summary.
#
# This helps reduce the amount of conversation history
# sent to the model and therefore helps control token usage.
summarization_middleware = SummarizationMiddleware(
    model=model,
    trigger=("tokens", 400),
    keep=("messages", 10)
)


# Creates an in-memory checkpointer.
#
# A checkpointer saves the agent's state between invocations.
# InMemorySaver stores this state in RAM.
#
# This is useful for maintaining conversation history
# associated with a particular thread_id.
#
# Since it is in memory, the stored data is lost when
# the Python program terminates.
checkpointer = InMemorySaver()


# Creates the LangChain agent.
#
# model:
# The LLM used by the agent.
#
# tools=[]:
# No external tools are being used in this example.
# The agent only communicates with the Groq model.
#
# middleware:
# Adds the summarization middleware to automatically
# manage long conversations.
#
# checkpointer:
# Enables the agent to save and restore conversation
# state using a thread ID.
agent = create_agent(
    model=model,
    tools=[],
    middleware=[summarization_middleware],
    checkpointer=checkpointer
)


# Creates a list of messages representing the conversation.
#
# Each dictionary contains:
# role    -> identifies who sent the message
# content -> contains the actual message text
#
# These messages are passed to the agent as conversation history.
messages = [
    {
        "role": "user",
        "content": "What is LangChain?"
    },
    {
        "role": "user",
        "content": "What is LangGraph?"
    },
    {
        "role": "user",
        "content": "Explain agents in LangChain."
    }
]


# Configuration passed to the agent during invocation.
#
# thread_id is extremely important when using a checkpointer.
# It acts as an identifier for a particular conversation.
#
# Messages invoked with the same thread_id belong to the
# same conversation state.
#
# For example:
# thread_001 -> Manasi's conversation
# thread_002 -> Another conversation
#
# Different thread IDs maintain separate conversation states.
config = {
    "configurable": {
        "thread_id": "thread_001"
    }
}


# Invokes the agent.
#
# {"messages": messages}
# passes the conversation messages to the agent.
#
# config=config
# tells LangGraph which conversation/thread should be used
# when saving and retrieving the checkpointed state.
response = agent.invoke(
    {
        "messages": messages
    },
    config=config
)


# The agent response contains a list of messages.
#
# response["messages"][-1]
# gets the last message, which is normally the latest AI response.
#
# .content
# extracts the actual text generated by the model.
print(response["messages"][-1].content)

In [2]:
len(response["messages"])

4

In [6]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.tools import tool

load_dotenv()

model = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)


# --------------------------------------------------
# HOTEL SEARCH TOOL
# --------------------------------------------------

@tool
def search_hotel(city: str) -> str:
    """Search for hotels in a given city."""
    
    hotels = {
        "mumbai": [
            "Taj Mahal Palace - ₹18,000/night",
            "Trident Nariman Point - ₹14,000/night",
            "The Oberoi Mumbai - ₹20,000/night"
        ],
        "pune": [
            "JW Marriott Pune - ₹12,000/night",
            "Conrad Pune - ₹13,000/night",
            "Hyatt Pune - ₹9,000/night"
        ],
        "delhi": [
            "The Leela Palace - ₹16,000/night",
            "ITC Maurya - ₹14,000/night",
            "Taj Palace - ₹15,000/night"
        ]
    }

    result = hotels.get(
        city.lower(),
        ["No hotels found for this city."]
    )

    return "\n".join(result)


# --------------------------------------------------
# SUMMARIZATION MIDDLEWARE
# --------------------------------------------------

summarization_middleware = SummarizationMiddleware(
    model=model,
    trigger=("tokens", 1000),
    keep=("messages", 4)
)


# --------------------------------------------------
# CHECKPOINTER
# --------------------------------------------------

checkpointer = InMemorySaver()


# --------------------------------------------------
# CREATE AGENT
# --------------------------------------------------

agent = create_agent(
    model=model,
    tools=[search_hotel],
    middleware=[summarization_middleware],
    checkpointer=checkpointer
)


# --------------------------------------------------
# CONFIG WITH THREAD ID
# --------------------------------------------------

config = {
    "configurable": {
        "thread_id": "hotel_search_001"
    }
}


# --------------------------------------------------
# CONVERSATION 1
# --------------------------------------------------

messages = [
    {
        "role": "user",
        "content": "Find hotels in Mumbai."
    }
]

response = agent.invoke(
    {
        "messages": messages
    },
    config=config
)

print("\n========== RESPONSE 1 ==========")
print(response["messages"][-1].content)


# --------------------------------------------------
# CONVERSATION 2
# --------------------------------------------------

response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Which one is the cheapest?"
            }
        ]
    },
    config=config
)

print("\n========== RESPONSE 2 ==========")
print(response["messages"][-1].content)


# --------------------------------------------------
# CONVERSATION 3
# --------------------------------------------------

response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What is the most luxurious option?"
            }
        ]
    },
    config=config
)

print("\n========== RESPONSE 3 ==========")
print(response["messages"][-1].content)


# --------------------------------------------------
# CONVERSATION 4
# --------------------------------------------------

response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Compare all the Mumbai hotels based on price, luxury,location, facilities and overall experience."
            }
        ]
    },
    config=config
)

print("\n========== RESPONSE 4 ==========")
print(response["messages"][-1].content)


# --------------------------------------------------
# SHOW MESSAGE HISTORY
# --------------------------------------------------

print("\n========== MESSAGE HISTORY ==========")

for i, message in enumerate(response["messages"]):
    print(f"\n{i + 1}. {type(message).__name__}")
    print(message.content)


# --------------------------------------------------
# CHECK TOKEN USAGE
# --------------------------------------------------

print("\n========== USAGE METADATA ==========")

for message in response["messages"]:
    if hasattr(message, "usage_metadata"):
        if message.usage_metadata:
            print(message.usage_metadata)


========== RESPONSE 1 ==========
Here are some popular hotels in Mumbai along with their approximate nightly rates (prices may vary by season, room type, and availability):

| Hotel | Approx. Rate (₹) | Highlights |
|-------|------------------|------------|
| **Taj Mahal Palace** | ₹18,000/night | Iconic heritage property, ocean‑front rooms, multiple fine‑dining restaurants, spa, and panoramic city views. |
| **Trident, Nariman Point** | ₹14,000/night | Contemporary luxury, rooftop pool, award‑winning restaurants, and a prime location near the business district. |
| **The Oberoi, Mumbai** | ₹20,000/night | Ultra‑luxury suites, world‑class service, spa, and a stunning view of the Arabian Sea. |

**Next steps:**

1. **Check Availability** – Let me know your travel dates and room preferences (single, double, suite, etc.) so I can fetch real‑time availability and rates.  
2. **Special Requests** – If you need anything specific (e.g., non‑smoking room, high floor, breakfast included, or ac

In [7]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.tools import tool

load_dotenv()

model = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)


@tool
def fraction_calculator(numerator: int, denominator: int) -> str:
    """Calculate and explain a fraction."""

    if denominator == 0:
        return "Denominator cannot be zero."

    decimal = numerator / denominator

    return (
        f"Fraction: {numerator}/{denominator}\n"
        f"Decimal value: {decimal:.4f}\n"
        f"Percentage: {decimal * 100:.2f}%"
    )


summarization_middleware = SummarizationMiddleware(
    model=model,
    trigger=("tokens", 1000),
    keep=("messages", 4)
)

checkpointer = InMemorySaver()

agent = create_agent(
    model=model,
    tools=[fraction_calculator],
    middleware=[summarization_middleware],
    checkpointer=checkpointer
)

config = {
    "configurable": {
        "thread_id": "fraction_agent_001"
    }
}


messages = [
    {
        "role": "user",
        "content": "Calculate 3/4 and explain its value."
    }
]

response = agent.invoke(
    {"messages": messages},
    config=config
)

print("\n========== RESPONSE 1 ==========")
print(response["messages"][-1].content)


response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Now compare 3/4 with 2/3."
            }
        ]
    },
    config=config
)

print("\n========== RESPONSE 2 ==========")
print(response["messages"][-1].content)


response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Which fraction represents the larger value?"
            }
        ]
    },
    config=config
)

print("\n========== RESPONSE 3 ==========")
print(response["messages"][-1].content)


print("\n========== MESSAGE HISTORY ==========")

for i, message in enumerate(response["messages"]):
    print(f"\n{i + 1}. {type(message).__name__}")
    print(message.content)


print("\n========== USAGE METADATA ==========")

for message in response["messages"]:
    if hasattr(message, "usage_metadata") and message.usage_metadata:
        print(message.usage_metadata)


========== RESPONSE 1 ==========
**3 ÷ 4 = 0.75**

- **Fraction form:** 3/4 (three parts out of four equal parts).  
- **Decimal form:** 0.75 (the fraction equals seventy‑five hundredths).  
- **Percentage form:** 75 % (three‑quarters of a whole).

**Why it’s 0.75**

When you divide 3 by 4, you’re asking how many times 4 fits into 3.  
- 4 goes into 3 zero times, so you start with 0.  
- Bring down a decimal point and add a zero: 30 ÷ 4 = 7 with a remainder of 2.  
- Bring down another zero: 20 ÷ 4 = 5 with no remainder.

Thus the decimal expansion stops at 0.75. In percentage terms, multiplying 0.75 by 100 gives 75 %. So 3/4 represents three‑quarters of a whole, or 75 % of it.

========== RESPONSE 2 ==========
### Comparison of \( \dfrac{3}{4} \) and \( \dfrac{2}{3} \)

| Method | Result |
|--------|--------|
| **Decimal conversion** | \( \dfrac{3}{4}=0.75 \) \( \dfrac{2}{3}\approx0.6667 \) |
| **Cross‑multiplication** | \(3 \times 3 = 9\) vs \(4 \times 2 = 8\) |
| **Common denominat

| Concept              | Code                           | What it measures             | Use                             |
| -------------------- | ------------------------------ | ---------------------------- | ------------------------------- |
| **Character length** | `len(text)`                    | Number of characters         | Check approximate text size     |
| **Input tokens**     | `usage["input_tokens"]`        | Tokens sent to model         | Monitor prompt/context usage    |
| **Output tokens**    | `usage["output_tokens"]`       | Tokens generated by model    | Monitor generated response size |
| **Total tokens**     | `usage["total_tokens"]`        | Input + output tokens        | Monitor total model usage       |
| **Input fraction**   | `input_tokens / total_tokens`  | Portion of usage from input  | Analyze context overhead        |
| **Output fraction**  | `output_tokens / total_tokens` | Portion of usage from output | Analyze generation overhead     |
